In [1]:
import pandas as pd
df = pd.read_csv('incidents_ml_processed.csv')

In [19]:
# Recreate the modeling table after a kernel or notebook reset.
threat_mapping = {'Oil': 0, 'Chemical': 1, 'Other': 2, 'Unknown': 2}
df['threat'] = df['threat'].map(threat_mapping).fillna(2).astype('int64')

boolean_columns = [
    'sheen_reported',
    'shoreline_impact',
    'wildlife_impact',
    'source_secured',
]
for column in boolean_columns:
    df[column] = df[column].astype('boolean')

df['source_is_vessel'] = df['source_type'].eq('Vessel')
df = df[[
    'threat',
    'reported_qty_gallons_equiv',
    'max_ptl_release_gallons',
    'agencies_involved',
    'actions_requested',
    'sheen_reported',
    'shoreline_impact',
    'wildlife_impact',
    'source_secured',
    'source_is_vessel',
]]

In [2]:
df.info()
#0 is oil, 1 is chemcial, 2 is other

<class 'pandas.DataFrame'>
RangeIndex: 4936 entries, 0 to 4935
Data columns (total 13 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Unnamed: 0                                  4936 non-null   int64  
 1   threat                                      4936 non-null   int64  
 2   reported_qty_gallons_equiv                  4936 non-null   float64
 3   max_ptl_release_gallons                     4936 non-null   float64
 4   sheen_reported                              4936 non-null   bool   
 5   shoreline_impact                            4936 non-null   bool   
 6   wildlife_impact                             4936 non-null   bool   
 7   source_secured                              4936 non-null   bool   
 8   source_is_vessel                            4936 non-null   bool   
 9   actions_requested_Overflight                4936 non-null   int64  
 10  actions_requested_Resou

In [32]:
df.to_csv('incidents_ml_processed.csv')

In [ ]:
vals = df['actions_requested'].value_counts()
frequent_actions = vals[vals > 50].index

action_dummies = pd.get_dummies(
    df['actions_requested'].where(df['actions_requested'].isin(frequent_actions)),
    prefix='actions_requested',
    dtype='int64',
)
df = pd.concat([df, action_dummies], axis=1)

print(f'One-hot encoded {len(frequent_actions)} frequent action values:')
print(list(frequent_actions))
df.head()

One-hot encoded 7 frequent action values:
['Trajectory Analysis', 'Sheen Assessment', 'Trajectory Analysis|Sheen Assessment', 'Resources at Risk Report', 'Overflight', 'Sheen Assessment|Overflight', 'Trajectory Analysis|Resources at Risk Report']


,threat,reported_qty_gallons_equiv,max_ptl_release_gallons,agencies_involved,actions_requested,sheen_reported,shoreline_impact,wildlife_impact,source_secured,source_is_vessel,actions_requested_Overflight,actions_requested_Resources at Risk Report,actions_requested_Sheen Assessment,actions_requested_Sheen Assessment|Overflight,actions_requested_Trajectory Analysis,actions_requested_Trajectory Analysis|Resources at Risk Report,actions_requested_Trajectory Analysis|Sheen Assessment
0,0,300.0,NaN,USCG|NOAA SSC|NOAA OR&R|Responsible Party/Owner,Worst-case Discharge Assessment,False,False,False,False,True,0,0,0,0,0,0,0
1,1,NaN,NaN,NOAA OR&R,ALOHA Modeling|Weather Forecast|Worst-case Dis...,False,False,False,False,False,0,0,0,0,0,0,0
2,0,NaN,NaN,USCG|NOAA SSC,Trajectory Analysis,False,False,False,False,True,0,0,0,0,1,0,0
3,0,2000.0,NaN,USCG|NOAA SSC,Sheen Assessment,True,False,False,False,True,0,0,1,0,0,0,0
4,0,NaN,NaN,USCG|NOAA OR&R|NOAA ERD,NaN,False,False,False,False,False,0,0,0,0,0,0,0


In [3]:
import re
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display

# Find the requested fields while allowing small naming differences in the dataset.
requested_fields = {
    'source_type': ['source_type', 'source type'],
    'threat': ['threat'],
    'commodity': ['commodity'],
    'agencies': ['agencies', 'agency', 'agencies_involved', 'agencies involved'],
    'actions_requested': ['actions_requested', 'actions requested', 'action_requested', 'action requested'],
}
normalized_columns = {re.sub(r'[^a-z0-9]', '', column.lower()): column for column in df.columns}
selected_columns = {}
for output_name, aliases in requested_fields.items():
    for alias in aliases:
        column = normalized_columns.get(re.sub(r'[^a-z0-9]', '', alias.lower()))
        if column:
            selected_columns[output_name] = column
            break

missing_fields = sorted(set(requested_fields) - set(selected_columns))
if missing_fields:
    raise KeyError(f'Missing requested columns: {missing_fields}. Available columns: {df.columns.tolist()}')

# Split list-like values so each agency/action is counted separately.
def explode_values(series):
    values = series.dropna().astype(str).str.split(r'\s*(?:;|\||,|/|\\)\s*').explode().str.strip()
    return values[values.ne('') & values.str.lower().ne('nan')]

count_tables = []
summary_rows = []
for output_name, column in selected_columns.items():
    counts = explode_values(df[column]).value_counts().rename_axis('value').reset_index(name='count')
    counts.insert(0, 'feature', output_name)
    count_tables.append(counts)
    top = counts.iloc[0]
    summary_rows.append({'feature': output_name, 'most_frequent_value': top['value'], 'count': int(top['count'])})

frequency_table = pd.concat(count_tables, ignore_index=True)
most_frequent_summary = pd.DataFrame(summary_rows)

print('Most frequent values')
display(most_frequent_summary)
print('All frequency counts')
display(frequency_table)

output_dir = Path('exports')
output_dir.mkdir(exist_ok=True)
most_frequent_summary.to_csv(output_dir / 'most_frequent_incident_features.csv', index=False)
frequency_table.to_csv(output_dir / 'incident_feature_frequency_counts.csv', index=False)

fig, axes = plt.subplots(3, 2, figsize=(16, 15))
axes = axes.ravel()
for axis, (feature, counts) in zip(axes, frequency_table.groupby('feature', sort=False)):
    top_counts = counts.head(10).sort_values('count')
    axis.barh(top_counts['value'], top_counts['count'], color='#1f77b4')
    axis.set_title(f'Top {feature.replace("_", " ").title()}')
    axis.set_xlabel('Number of incidents')
    axis.tick_params(axis='y', labelsize=8)
for axis in axes[len(selected_columns):]:
    axis.set_visible(False)
fig.suptitle('Most Frequent Incident Features', fontsize=16)
fig.tight_layout()
fig.savefig(output_dir / 'most_frequent_incident_features.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'Exported tables and visualization to: {output_dir.resolve()}')

KeyError: "Missing requested columns: ['actions_requested', 'agencies', 'commodity', 'source_type']. Available columns: ['Unnamed: 0', 'threat', 'reported_qty_gallons_equiv', 'max_ptl_release_gallons', 'sheen_reported', 'shoreline_impact', 'wildlife_impact', 'source_secured', 'source_is_vessel', 'actions_requested_Overflight', 'actions_requested_Resources at Risk Report', 'actions_requested_Sheen Assessment', 'actions_requested_Trajectory Analysis']"

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4936 entries, 0 to 4935
Data columns (total 12 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   threat                                      4936 non-null   int64  
 1   reported_qty_gallons_equiv                  4936 non-null   float64
 2   max_ptl_release_gallons                     4936 non-null   float64
 3   sheen_reported                              4936 non-null   boolean
 4   shoreline_impact                            4936 non-null   boolean
 5   wildlife_impact                             4936 non-null   boolean
 6   source_secured                              4936 non-null   boolean
 7   source_is_vessel                            4936 non-null   bool   
 8   actions_requested_Overflight                4936 non-null   int64  
 9   actions_requested_Resources at Risk Report  4936 non-null   int64  
 10  actions_requested_Sheen

In [35]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier


def run_action_model_comparison(data, target, random_state=42):
    """Compare models and rank them by a balanced combination of accuracy and F1."""
    if target not in data.columns:
        raise KeyError(f"Target column {target!r} was not found. Available columns: {data.columns.tolist()}")

    action_columns = [column for column in data.columns if column.startswith('actions_requested_')]
    secured_columns = [column for column in data.columns if 'secured' in column.lower()]
    excluded_columns = set(action_columns + secured_columns + [target])
    feature_columns = [column for column in data.columns if column not in excluded_columns]
    model_data = data[feature_columns + [target]].copy().dropna(subset=[target])
    y = model_data[target].astype('int64')
    if y.nunique() != 2:
        raise ValueError(f"{target} must contain exactly two classes, found {sorted(y.unique())}")

    X = model_data[feature_columns]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    def replicate_classes(features, labels, seed):
        """Oversample each binary class to the size of the larger class."""
        combined = features.copy()
        combined['__target__'] = labels.to_numpy()
        target_size = labels.value_counts().max()
        class_frames = []
        for class_value in sorted(labels.unique()):
            class_frame = combined[combined['__target__'] == class_value]
            class_frames.append(class_frame.sample(
                n=target_size,
                replace=len(class_frame) < target_size,
                random_state=seed + int(class_value),
            ))
        balanced = pd.concat(class_frames).sample(frac=1, random_state=seed).reset_index(drop=True)
        return balanced.drop(columns='__target__'), balanced['__target__'].astype('int64')

    train_counts_before = y_train.value_counts().sort_index().to_dict()
    test_counts_before = y_test.value_counts().sort_index().to_dict()
    X_train, y_train = replicate_classes(X_train, y_train, random_state)
    X_test, y_test = replicate_classes(X_test, y_test, random_state + 1)
    train_counts_after = y_train.value_counts().sort_index().to_dict()
    test_counts_after = y_test.value_counts().sort_index().to_dict()

    numeric_columns = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    categorical_columns = X.select_dtypes(exclude=['number', 'bool']).columns.tolist()
    numeric_pipeline = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('scale', StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    preprocessor = ColumnTransformer([
        ('numeric', numeric_pipeline, numeric_columns),
        ('categorical', categorical_pipeline, categorical_columns),
    ])
    models = {
        'Logistic regression': LogisticRegression(
            l1_ratio=0, solver='lbfgs', max_iter=2000, class_weight='balanced'
        ),
        'Logistic regression (lasso)': LogisticRegression(
            l1_ratio=1, solver='saga', max_iter=5000, class_weight='balanced', random_state=random_state
        ),
        'XGBoost': XGBClassifier(
            n_estimators=250, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
            random_state=random_state, n_jobs=2,
        ),
    }

    scores = []
    fitted_models = {}
    for model_name, estimator in models.items():
        pipeline = Pipeline([('preprocess', preprocessor), ('model', estimator)])
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        probabilities = pipeline.predict_proba(X_test)[:, 1]
        accuracy = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions, zero_division=0)
        # Harmonic mean makes a low F1 materially reduce the selection score.
        accuracy_f1_score = (
            2 * accuracy * f1 / (accuracy + f1) if accuracy + f1 else 0.0
        )
        scores.append({
            'model': model_name,
            'accuracy': accuracy,
            'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
            'f1': f1,
            'accuracy_f1_score': accuracy_f1_score,
            'roc_auc': roc_auc_score(y_test, probabilities),
        })
        fitted_models[model_name] = pipeline

    scores = pd.DataFrame(scores).sort_values(
        ['accuracy_f1_score', 'f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)
    winning_name = scores.loc[0, 'model']
    winning_pipeline = fitted_models[winning_name]

    permutation = permutation_importance(
        winning_pipeline, X_test, y_test, scoring='f1',
        n_repeats=10, random_state=random_state, n_jobs=2,
    )
    feature_importance = pd.DataFrame({
        'feature': X_test.columns,
        'importance_mean': permutation.importances_mean,
        'importance_std': permutation.importances_std,
    }).sort_values('importance_mean', ascending=False).reset_index(drop=True)

    print(f'Target: {target} (1 means the target event was present)')
    print(f'Positive class rate: {y.mean():.1%}')
    print(f'Class counts before replication - train: {train_counts_before}, test: {test_counts_before}')
    print(f'Class counts after replication - train: {train_counts_after}, test: {test_counts_after}')
    print(f'Excluded secured/action predictors: {sorted(excluded_columns.intersection(data.columns))}')
    print('Winner selected by the accuracy-F1 harmonic mean; low-F1 models are penalized.')
    print(f'Winning model: {winning_name}')
    print('Model comparison on the replicated held-out test set')
    display(scores)
    print('Top predictive associations by permutation importance (F1 decrease)')
    display(feature_importance.head(20))

    if hasattr(winning_pipeline.named_steps['model'], 'coef_'):
        transformed_names = winning_pipeline.named_steps['preprocess'].get_feature_names_out()
        coefficients = winning_pipeline.named_steps['model'].coef_[0]
        direction = pd.DataFrame({
            'transformed_feature': transformed_names,
            'coefficient': coefficients,
            'odds_ratio': np.exp(coefficients),
        })
        print('Signed associations for the winning logistic model: positive values increase predicted event odds')
        display(direction.reindex(direction['coefficient'].abs().sort_values(ascending=False).index).head(20))

    return {
        'target': target,
        'scores': scores,
        'winning_model_name': winning_name,
        'winning_pipeline': winning_pipeline,
        'feature_importance': feature_importance,
        'test_data': (X_test, y_test),
    }


In [36]:
# Predict whether the incident source was secured from the remaining incident features.
# Action-request columns are excluded by the shared helper to prevent leakage.
source_secured_results = run_action_model_comparison(
    df, 'source_secured'
)

Target: source_secured (1 means the target event was present)
Positive class rate: 9.3%
Class counts before replication - train: {0: 3582, 1: 366}, test: {0: 896, 1: 92}
Class counts after replication - train: {0: 3582, 1: 3582}, test: {0: 896, 1: 896}
Excluded secured/action predictors: ['actions_requested_Overflight', 'actions_requested_Resources at Risk Report', 'actions_requested_Sheen Assessment', 'actions_requested_Trajectory Analysis', 'source_secured']
Winner selected by the accuracy-F1 harmonic mean; low-F1 models are penalized.
Winning model: Logistic regression
Model comparison on the replicated held-out test set


,model,accuracy,balanced_accuracy,f1,accuracy_f1_score,roc_auc
0,Logistic regression,0.608817,0.608817,0.612064,0.610436,0.626552
1,Logistic regression (lasso),0.608817,0.608817,0.612064,0.610436,0.626607
2,XGBoost,0.612723,0.612723,0.551100,0.580280,0.646391


Top predictive associations by permutation importance (F1 decrease)


,feature,importance_mean,importance_std
0,threat,0.061804,0.005481
1,source_is_vessel,0.053594,0.013174
2,reported_qty_gallons_equiv,0.041602,0.005995
3,max_ptl_release_gallons,0.017562,0.003605
4,sheen_reported,0.004287,0.002051
5,shoreline_impact,-0.004015,0.001799
6,wildlife_impact,-0.004462,0.003139


Signed associations for the winning logistic model: positive values increase predicted event odds


,transformed_feature,coefficient,odds_ratio
6,numeric__source_is_vessel,-0.310505,0.733077
1,numeric__reported_qty_gallons_equiv,-0.285124,0.751921
0,numeric__threat,0.187679,1.206446
4,numeric__shoreline_impact,0.135715,1.145356
2,numeric__max_ptl_release_gallons,-0.056520,0.945048
5,numeric__wildlife_impact,0.054858,1.056391
3,numeric__sheen_reported,-0.009110,0.990931


In [31]:
# Predict whether Overflight is requested from incident features, excluding every action column.
overflight_results = run_action_model_comparison(
    df, 'actions_requested_Overflight'
)

Target: actions_requested_Overflight (1 means the target event was present)
Positive class rate: 4.7%
Class counts before replication - train: {0: 3762, 1: 186}, test: {0: 942, 1: 46}
Class counts after replication - train: {0: 3762, 1: 3762}, test: {0: 942, 1: 942}
Excluded secured/action predictors: ['actions_requested_Overflight', 'actions_requested_Resources at Risk Report', 'actions_requested_Sheen Assessment', 'actions_requested_Trajectory Analysis', 'source_secured']
Winner selected by the accuracy-F1 harmonic mean; low-F1 models are penalized.
Winning model: Logistic regression
Model comparison on the replicated held-out test set


,model,accuracy,balanced_accuracy,f1,accuracy_f1_score,roc_auc
0,Logistic regression,0.703291,0.703291,0.736445,0.719486,0.805909
1,Logistic regression (lasso),0.703291,0.703291,0.736445,0.719486,0.806033
2,XGBoost,0.684183,0.684183,0.667412,0.675693,0.731733


Top predictive associations by permutation importance (F1 decrease)


,feature,importance_mean,importance_std
0,sheen_reported,0.096041,0.003335
1,threat,0.067703,0.005746
2,shoreline_impact,-0.001009,0.001296
3,wildlife_impact,-0.001117,0.001672
4,reported_qty_gallons_equiv,-0.009973,0.001714
5,max_ptl_release_gallons,-0.013881,0.005225
6,source_is_vessel,-0.016001,0.001605


Signed associations for the winning logistic model: positive values increase predicted event odds


,transformed_feature,coefficient,odds_ratio
3,numeric__sheen_reported,0.694044,2.001794
0,numeric__threat,0.519010,1.680363
5,numeric__wildlife_impact,0.137229,1.147090
2,numeric__max_ptl_release_gallons,0.099319,1.104419
4,numeric__shoreline_impact,0.037302,1.038007
1,numeric__reported_qty_gallons_equiv,-0.033222,0.967324
6,numeric__source_is_vessel,-0.018612,0.981561


In [32]:
# Predict whether Sheen Assessment is requested from incident features, excluding every action column.
sheen_assessment_results = run_action_model_comparison(
    df, 'actions_requested_Sheen Assessment'
)

Target: actions_requested_Sheen Assessment (1 means the target event was present)
Positive class rate: 15.0%
Class counts before replication - train: {0: 3355, 1: 593}, test: {0: 840, 1: 148}
Class counts after replication - train: {0: 3355, 1: 3355}, test: {0: 840, 1: 840}
Excluded secured/action predictors: ['actions_requested_Overflight', 'actions_requested_Resources at Risk Report', 'actions_requested_Sheen Assessment', 'actions_requested_Trajectory Analysis', 'source_secured']
Winner selected by the accuracy-F1 harmonic mean; low-F1 models are penalized.
Winning model: Logistic regression
Model comparison on the replicated held-out test set


,model,accuracy,balanced_accuracy,f1,accuracy_f1_score,roc_auc
0,Logistic regression,0.979762,0.979762,0.980163,0.979963,0.981731
1,Logistic regression (lasso),0.979762,0.979762,0.980163,0.979963,0.981813
2,XGBoost,0.979762,0.979762,0.980163,0.979963,0.981245


Top predictive associations by permutation importance (F1 decrease)


,feature,importance_mean,importance_std
0,sheen_reported,0.463711,0.008578
1,reported_qty_gallons_equiv,0.000000,0.000000
2,threat,0.000000,0.000000
3,max_ptl_release_gallons,0.000000,0.000000
4,shoreline_impact,0.000000,0.000000
5,wildlife_impact,0.000000,0.000000
6,source_is_vessel,0.000000,0.000000


Signed associations for the winning logistic model: positive values increase predicted event odds


,transformed_feature,coefficient,odds_ratio
3,numeric__sheen_reported,5.308483,202.043497
5,numeric__wildlife_impact,-0.260754,0.770470
0,numeric__threat,0.227314,1.255224
1,numeric__reported_qty_gallons_equiv,0.172429,1.188187
4,numeric__shoreline_impact,-0.081963,0.921306
2,numeric__max_ptl_release_gallons,-0.073094,0.929514
6,numeric__source_is_vessel,-0.062980,0.938963


In [33]:
# Predict whether Trajectory Analysis is requested from incident features, excluding every action column.
trajectory_analysis_results = run_action_model_comparison(
    df, 'actions_requested_Trajectory Analysis'
)

Target: actions_requested_Trajectory Analysis (1 means the target event was present)
Positive class rate: 18.7%
Class counts before replication - train: {0: 3208, 1: 740}, test: {0: 803, 1: 185}
Class counts after replication - train: {0: 3208, 1: 3208}, test: {0: 803, 1: 803}
Excluded secured/action predictors: ['actions_requested_Overflight', 'actions_requested_Resources at Risk Report', 'actions_requested_Sheen Assessment', 'actions_requested_Trajectory Analysis', 'source_secured']
Winner selected by the accuracy-F1 harmonic mean; low-F1 models are penalized.
Winning model: XGBoost
Model comparison on the replicated held-out test set


,model,accuracy,balanced_accuracy,f1,accuracy_f1_score,roc_auc
0,XGBoost,0.689290,0.689290,0.694427,0.691849,0.744172
1,Logistic regression,0.666874,0.666874,0.712520,0.688942,0.714068
2,Logistic regression (lasso),0.666874,0.666874,0.712520,0.688942,0.713947


Top predictive associations by permutation importance (F1 decrease)


,feature,importance_mean,importance_std
0,threat,0.139290,0.011255
1,max_ptl_release_gallons,0.036119,0.006211
2,sheen_reported,0.022724,0.004877
3,source_is_vessel,0.022502,0.004832
4,reported_qty_gallons_equiv,0.022174,0.006077
5,wildlife_impact,0.006920,0.003958
6,shoreline_impact,0.000929,0.000931


In [34]:
# Predict whether Resources at Risk Report is requested from incident features, excluding every action column.
resources_at_risk_results = run_action_model_comparison(
    df, 'actions_requested_Resources at Risk Report'
)

Target: actions_requested_Resources at Risk Report (1 means the target event was present)
Positive class rate: 4.4%
Class counts before replication - train: {0: 3774, 1: 174}, test: {0: 945, 1: 43}
Class counts after replication - train: {0: 3774, 1: 3774}, test: {0: 945, 1: 945}
Excluded secured/action predictors: ['actions_requested_Overflight', 'actions_requested_Resources at Risk Report', 'actions_requested_Sheen Assessment', 'actions_requested_Trajectory Analysis', 'source_secured']
Winner selected by the accuracy-F1 harmonic mean; low-F1 models are penalized.
Winning model: Logistic regression
Model comparison on the replicated held-out test set


,model,accuracy,balanced_accuracy,f1,accuracy_f1_score,roc_auc
0,Logistic regression,0.730688,0.730688,0.770617,0.750122,0.761000
1,Logistic regression (lasso),0.730688,0.730688,0.770617,0.750122,0.760726
2,XGBoost,0.671958,0.671958,0.676071,0.674008,0.720330


Top predictive associations by permutation importance (F1 decrease)


,feature,importance_mean,importance_std
0,threat,1.302169e-01,0.004384
1,sheen_reported,7.185490e-02,0.002426
2,shoreline_impact,7.913067e-03,0.000827
3,max_ptl_release_gallons,2.429875e-04,0.000159
4,reported_qty_gallons_equiv,-6.260143e-08,0.000220
5,source_is_vessel,-6.948759e-05,0.000139
6,wildlife_impact,-3.474380e-04,0.000000


Signed associations for the winning logistic model: positive values increase predicted event odds


,transformed_feature,coefficient,odds_ratio
3,numeric__sheen_reported,-2.168491,0.114350
0,numeric__threat,-0.671350,0.511018
4,numeric__shoreline_impact,-0.532254,0.587280
6,numeric__source_is_vessel,0.098277,1.103268
1,numeric__reported_qty_gallons_equiv,-0.076848,0.926031
5,numeric__wildlife_impact,0.076264,1.079247
2,numeric__max_ptl_release_gallons,-0.053762,0.947658


In [37]:
from pathlib import Path

export_dir = Path('exports')
export_dir.mkdir(exist_ok=True)
model_results = {
    'source_secured': source_secured_results,
    'overflight': overflight_results,
    'sheen_assessment': sheen_assessment_results,
    'trajectory_analysis': trajectory_analysis_results,
    'resources_at_risk': resources_at_risk_results,
}

score_tables = []
importance_tables = []
for target_name, result in model_results.items():
    scores = result['scores'].copy()
    scores.insert(0, 'target', target_name)
    score_tables.append(scores)

    importance = result['feature_importance'].copy()
    importance.insert(0, 'target', target_name)
    importance_tables.append(importance)

    scores.to_csv(export_dir / f'{target_name}_model_scores.csv', index=False)
    importance.to_csv(export_dir / f'{target_name}_feature_importance.csv', index=False)

all_scores = pd.concat(score_tables, ignore_index=True)
all_importance = pd.concat(importance_tables, ignore_index=True)
all_scores.to_csv(export_dir / 'all_model_scores.csv', index=False)
all_importance.to_csv(export_dir / 'all_feature_importance.csv', index=False)

print(f'Exported model score tables to: {export_dir / "all_model_scores.csv"}')
print(f'Exported feature-importance tables to: {export_dir / "all_feature_importance.csv"}')
print(f'Exported {len(model_results) * 2} per-target tables to: {export_dir.resolve()}')


Exported model score tables to: exports/all_model_scores.csv
Exported feature-importance tables to: exports/all_feature_importance.csv
Exported 10 per-target tables to: /mnt/d/Fall 2026/Data-Analysis-and-Forecasting-for-Oil-companies-insurance-s-risk-management-/incidentNews dataset/exports
